# NB-03 | Benchmark: Baseline 3 — Props LoRA

Оценка **Anima Base + Turbo LoRA + Props LoRA** (@spll_icn trigger).  
Датасет: 288 реальных Painterly Spell Icons.

**Важно:** reference set здесь `reference/props`, а не `reference/characters`!  
KID считается относительно Props-референсов, иначе сравнение будет некорректным.

In [1]:
import os, json, struct
from pathlib import Path
import torch, numpy as np, mlflow
from PIL import Image
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
import open_clip
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [2]:
BASELINE_ID   = 'baseline_3'
GENERATED_DIR = Path('generated/baseline_3')
REFERENCE_DIR = Path('reference/props')   # <-- важно: props, не characters!

PROPS_LORA_PATH = Path('ComfyUI/models/loras/Anima/spll_icn_props_v1.safetensors')

MLFLOW_URI      = 'http://127.0.0.1:5000'
EXPERIMENT_NAME = 'anima_baseline_benchmarks'

MODEL_PARAMS = {
    'model.name':          'anima_base',
    'model.family':        'cosmos2_dit',
    'model.type':          'props_lora',
    'model.diffusion':     'animaBase_v1.safetensors',
    'model.text_encoder':  'qwen_3_06b_base.safetensors',
    'model.vae':           'qwen_image_vae.safetensors',
    'model.props_lora':    'spll_icn_props_v1.safetensors',
    'model.lora_stack':    'turbo+props',
    'model.trigger':       '@spll_icn',
    'model.dataset_size':  288,
    'model.dataset_type':  'real_data_painterly_icons',
    'model.caption_type':  'hybrid_booru_wd14_plus_llava7b_nlp',
    'model.metadata_source': 'config_known',
}

INFER_PARAMS = {
    'infer.baseline_id':    BASELINE_ID,
    'infer.sampler':        'euler_ancestral',
    'infer.scheduler':      'normal',
    'infer.steps':          12,
    'infer.cfg':            2.0,
    'infer.width':          512,
    'infer.height':         512,
    'infer.turbo_strength': 1.0,
    'infer.style_strength': 0.85,
    'infer.llm_expansion':  False,
    'infer.seed_policy':    'idx*7919',
    'infer.prompt_set':     'props_prompts_v1',
    'infer.n_images':       100,
}

DATA_PARAMS = {
    'data.generated_dir':  str(GENERATED_DIR),
    'data.reference_dir':  str(REFERENCE_DIR),
    'data.reference_type': 'painterly_spell_icons_288',
    'data.mode_collapse_risk': 'elevated_small_dataset',
}

print('Config loaded ✓')

Config loaded ✓


In [3]:
def load_t(folder, size=(299,299), limit=None):
    files = sorted(folder.glob('*.png'))
    if limit: files = files[:limit]
    return torch.stack([torch.tensor(np.array(Image.open(p).convert('RGB').resize(size))).permute(2,0,1) for p in files])

def load_pil(folder, limit=None):
    files = sorted(folder.glob('*.png'))
    if limit: files = files[:limit]
    return [Image.open(p).convert('RGB') for p in files]

def make_grid(imgs, n=8, thumb=128):
    imgs = [i.resize((thumb,thumb)) for i in imgs[:n]]
    c = Image.new('RGB', (thumb*len(imgs), thumb), (20,20,20))
    for i,im in enumerate(imgs): c.paste(im, (thumb*i, 0))
    return c

def read_meta(path):
    try:
        with open(path,'rb') as f:
            n = struct.unpack('<Q', f.read(8))[0]
            return json.loads(f.read(n).decode('utf-8')).get('__metadata__',{})
    except Exception as e:
        return {'error': str(e)}

def compute_kid(gen, ref, subset=50):
    k = KernelInceptionDistance(subset_size=subset, normalize=True).to(DEVICE)
    k.update(load_t(ref).to(DEVICE), real=True)
    k.update(load_t(gen).to(DEVICE), real=False)
    m,s = k.compute(); return float(m.cpu()), float(s.cpu())

def compute_clip(gen, prompts):
    model,_,prep = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
    model = model.to(DEVICE).eval()
    tok = open_clip.get_tokenizer('ViT-B-32')
    sc = []
    for i,p in enumerate(tqdm(sorted(gen.glob('*.png')),'CLIP')):
        if i>=len(prompts): break
        img = prep(Image.open(p).convert('RGB')).unsqueeze(0).to(DEVICE)
        txt = tok([prompts[i]]).to(DEVICE)
        with torch.no_grad():
            a=model.encode_image(img); a/=a.norm(dim=-1,keepdim=True)
            b=model.encode_text(txt);  b/=b.norm(dim=-1,keepdim=True)
            sc.append(float((a*b).sum().cpu()))
    return float(np.mean(sc))

def compute_lpips(gen, n_pairs=200):
    lp = LearnedPerceptualImagePatchSimilarity(net_type='vgg', normalize=True).to(DEVICE)
    files = sorted(gen.glob('*.png'))
    idx = np.random.default_rng(42).integers(0, len(files), size=(n_pairs,2))
    sc = []
    def tt(p): return torch.tensor(np.array(Image.open(p).convert('RGB').resize((256,256))).astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(DEVICE)
    for a,b in tqdm(idx,'LPIPS'):
        if a==b: continue
        with torch.no_grad(): sc.append(float(lp(tt(files[a]), tt(files[b])).cpu()))
    return float(np.mean(sc))

print('Helpers ready ✓')

Helpers ready ✓


In [4]:
PROPS_PROMPTS_BASE = [
    'glowing magic orb, blue energy, dark background, no humans',
    'fire spell icon, orange glow, dark background, no humans',
    'ice crystal, cyan light, dark background, no humans',
    'lightning rune, yellow glow, dark background, no humans',
    'healing potion, green glow, dark background, no humans',
    'dark magic circle, purple energy, dark background, no humans',
    'glowing sword, blue outline, dark background, no humans',
    'fire aura, red orange, simple background, no humans',
    'water elemental, blue mist, dark background, no humans',
    'magic crystal, pink glow, dark background, no humans',
]
PROMPTS_100 = [PROPS_PROMPTS_BASE[i % 10] for i in range(100)]
print(f'Prompts: {len(PROMPTS_100)}')

Prompts: 100


In [5]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f'eval_{BASELINE_ID}_propsLoRA_cfg{INFER_PARAMS["infer.cfg"]}_w{INFER_PARAMS["infer.style_strength"]}'

with mlflow.start_run(run_name=run_name) as run:
    print(f'Run ID: {run.info.run_id}')

    mlflow.set_tags({
        'evaluation_type': 'benchmark',
        'training_logged_retroactively': 'true',
        'family':          'props',
        'baseline_id':     BASELINE_ID,
        'environment':     'local_rtx4060ti_16gb',
        'framework':       'comfyui',
    })

    mlflow.log_params(MODEL_PARAMS)
    mlflow.log_params(INFER_PARAMS)
    mlflow.log_params(DATA_PARAMS)

    # Metadata из safetensors Props LoRA
    if PROPS_LORA_PATH.exists():
        meta = read_meta(PROPS_LORA_PATH)
        Path('tmp').mkdir(parents=True, exist_ok=True)
        mp = f'tmp/{BASELINE_ID}_lora_meta.json'
        with open(mp,'w') as f: json.dump(meta, f, indent=2)
        mlflow.log_artifact(mp, artifact_path='model_provenance')
        for k in ['ss_network_dim','ss_network_alpha','ss_learning_rate','ss_epoch','ss_optimizer']:
            if k in meta: mlflow.log_param(f'train.{k.replace("ss_","")}', meta[k])
    else:
        print('Props LoRA file not found — metadata skipped')

    print('Computing metrics...')
    kid_mean, kid_std = compute_kid(GENERATED_DIR, REFERENCE_DIR)
    clip_score        = compute_clip(GENERATED_DIR, PROMPTS_100)
    lpips_div         = compute_lpips(GENERATED_DIR)

    # Предупреждение: малый датасет 288 штук — следим за LPIPS
    if lpips_div < 0.10:
        mlflow.set_tag('warning', 'possible_mode_collapse_lpips_below_0.10')
        print('⚠ WARNING: LPIPS low — possible mode collapse!')

    mlflow.log_metrics({
        'eval.kid_mean':        kid_mean,
        'eval.kid_std':         kid_std,
        'eval.clip_score':      clip_score,
        'eval.lpips_diversity': lpips_div,
        'eval.n_generated':     float(len(list(GENERATED_DIR.glob('*.png')))),
        'eval.n_reference':     float(len(list(REFERENCE_DIR.glob('*.png')))),
    })

    grid = make_grid(load_pil(GENERATED_DIR, limit=8))
    gp = f'tmp/{BASELINE_ID}_grid.png'; grid.save(gp)
    mlflow.log_artifact(gp, artifact_path='samples')

    print(f'\n=== BL3 Results ===')
    print(f'KID:   {kid_mean:.6f} ± {kid_std:.6f}')
    print(f'CLIP:  {clip_score:.4f}')
    print(f'LPIPS: {lpips_div:.4f}')
    print('✓ Logged to MLflow')

Run ID: 90f448f0279844c4a7ac283c7a6afab6
Props LoRA file not found — metadata skipped
Computing metrics...


e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Metric `Kernel Inception Distance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
LPIPS: 100%|██████████| 200/200 [01:03<00:00,  3.15it/s]



=== BL3 Results ===
KID:   0.038077 ± 0.004436
CLIP:  0.3196
LPIPS: 0.6983
✓ Logged to MLflow
🏃 View run eval_baseline_3_propsLoRA_cfg2.0_w0.85 at: http://127.0.0.1:5000/#/experiments/1/runs/90f448f0279844c4a7ac283c7a6afab6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
